<a href="https://colab.research.google.com/github/cras-lab/LangChain/blob/main/MCP_%EC%9E%91%EB%8F%99%EC%9B%90%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

먼저 필요한 환경 변수를 설정한다.

In [7]:
LLM_PROVIDER = "openai" # claude 또는 gemini
KEYFROM_USERDATA = True  # 외부에서 입력하려면 False 로
TEMPERATURE = 0.2

필요한 모듈을 설치한다.

In [2]:
!pip -q install -U langchain langchain-openai langchain-anthropic langchain-google-genai langchain-core pandas
import json
from getpass import getpass
import pandas as pd
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

try:
    from langchain.tools import tool
except Exception:
    from langchain_core.tools import tool

Legacy PART: 가상의 은행 데이터를 만든다.<BR>
이 부분은 서버 사이드이다.

In [3]:
# ============================================================
# 가상 은행 데이터
# ============================================================

ACCOUNTS = {
    "1001": {
        "owner": "Kim",
        "balance": 1_250_000,
        "currency": "KRW",
    },
    "1002": {
        "owner": "Lee",
        "balance": 870_000,
        "currency": "KRW",
    },
    "1003": {
        "owner": "Park",
        "balance": 4_320_000,
        "currency": "KRW",
    },
}

TRANSACTIONS = {
    "1001": [
        {"date": "2026-07-01", "merchant": "Starbucks", "amount": -6_200, "category": "cafe"},
        {"date": "2026-07-01", "merchant": "Salary", "amount": 3_000_000, "category": "income"},
        {"date": "2026-07-02", "merchant": "Costco", "amount": -154_000, "category": "shopping"},
        {"date": "2026-07-03", "merchant": "Bookstore", "amount": -42_000, "category": "book"},
        {"date": "2026-07-03", "merchant": "Transfer to Friend", "amount": -80_000, "category": "transfer"},
    ],
    "1002": [
        {"date": "2026-07-02", "merchant": "Subway", "amount": -9_500, "category": "food"},
        {"date": "2026-07-03", "merchant": "Transfer", "amount": 500_000, "category": "income"},
    ],
    "1003": [
        {"date": "2026-07-01", "merchant": "Rent", "amount": -900_000, "category": "housing"},
        {"date": "2026-07-02", "merchant": "Hospital", "amount": -120_000, "category": "medical"},
        {"date": "2026-07-03", "merchant": "Dividend", "amount": 75_000, "category": "income"},
    ],
}


def get_balance_data(account_id: str) -> dict:
    if account_id not in ACCOUNTS:
        raise ValueError("account not found")

    account = ACCOUNTS[account_id]

    return {
        "account_id": account_id,
        "owner": account["owner"],
        "balance": account["balance"],
        "currency": account["currency"],
    }


def list_transactions_data(account_id: str, min_abs_amount: int = 0) -> list[dict]:
    if account_id not in TRANSACTIONS:
        raise ValueError("account not found")

    return [
        tx for tx in TRANSACTIONS[account_id]
        if abs(tx["amount"]) >= min_abs_amount
    ]


def summarize_account_data(account_id: str, min_abs_amount: int = 30_000) -> dict:
    if account_id not in ACCOUNTS:
        raise ValueError("account not found")

    balance = get_balance_data(account_id)
    transactions = list_transactions_data(account_id, min_abs_amount=min_abs_amount)

    total_income = sum(
        tx["amount"] for tx in TRANSACTIONS[account_id]
        if tx["amount"] > 0
    )

    total_spending = -sum(
        tx["amount"] for tx in TRANSACTIONS[account_id]
        if tx["amount"] < 0
    )

    return {
        "balance": balance,
        "large_transactions": transactions,
        "total_income": total_income,
        "total_spending": total_spending,
    }

가상의 은행의 MCP 서버를 구축한다.

In [4]:
@tool
def get_balance(account_id: str) -> dict:
    """
    계좌 ID를 받아 현재 잔액, 예금주, 통화 정보를 조회한다.
    잔고, 잔액, 예금주 확인 요청에 사용한다.

    Args:
        account_id: 조회할 계좌 ID. 예: 1001
    """
    return get_balance_data(account_id)


@tool
def list_transactions(account_id: str, min_abs_amount: int = 30000) -> dict:
    """
    계좌 ID와 최소 거래금액을 받아 거래내역을 조회한다.
    거래내역, 입금, 출금, 지출, 고액 거래, 큰돈이 나간 거래를 확인할 때 사용한다.
    사용자가 '큰 거래' 또는 '고액 거래'라고만 말하고 금액 기준을 주지 않으면 min_abs_amount=30000을 사용한다.

    Args:
        account_id: 조회할 계좌 ID. 예: 1001
        min_abs_amount: 절대값 기준 최소 거래금액. 예: 30000
    """
    return {
        "account_id": account_id,
        "transactions": list_transactions_data(account_id, min_abs_amount),
    }


@tool
def summarize_account(account_id: str, min_abs_amount: int = 30000) -> dict:
    """
    계좌의 현재 잔액, 총 입금, 총 지출, 일정 금액 이상 거래를 함께 요약한다.
    사용자가 계좌 상태를 전체적으로 봐달라거나 소비 패턴을 요약해달라고 할 때 사용한다.
    금액 기준이 없으면 min_abs_amount=30000을 사용한다.

    Args:
        account_id: 조회할 계좌 ID. 예: 1001
        min_abs_amount: 큰 거래로 볼 최소 금액. 예: 30000
    """
    return summarize_account_data(account_id, min_abs_amount)


TOOLS = [
    get_balance,
    list_transactions,
    summarize_account,
]

TOOL_MAP = {
    tool_obj.name: tool_obj
    for tool_obj in TOOLS
}

여기서 부터 실제 클라이언트 쪽이다.

키를 입력받는다. 자신의 보관키 이름에 맞춰 수정

In [5]:
if KEYFROM_USERDATA:
  from google.colab import userdata
  API_KEY = userdata.get('OPENAI').strip() # 자신의 보관키에 따라 수정
else:
  import getpass
  API_KEY = getpass.getpass('Enter API key: ')

LLM을 생성한다.

In [8]:
if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(
        model="gpt-5-nano",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )

elif LLM_PROVIDER == "claude":
    from langchain_anthropic import ChatAnthropic

    llm = ChatAnthropic(
        model="claude-sonnet-4-6",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )


elif LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )

TOOL을 읽어온다. 실제 환경에서는 TOOLS라는 변수가 존재할리 없다.<BR>
실제는 tools/list 메서드가 서버에 정의되고 LangChain 의 경우 tools = await client.get_tools() 등으로 서버에서 읽어온다.

In [9]:
tool_rows = []

for tool_obj in TOOLS:
    tool_rows.append({
        "name": tool_obj.name,
        "description": tool_obj.description,
        "args_schema": tool_obj.args,
    })

읽어온 툴을 출력해 보자

In [ ]:
display(pd.DataFrame(tool_rows))

tool에서 name / description / argument schema를 뽑아 LLM API가 이해하는 tool-calling JSON schema 형태로 가공해서 이후 LLM 호출에 함께 붙이는 단계

In [11]:
llm_with_tools = llm.bind_tools(TOOLS)

llm_with_tools를 한번 출력해 보자.

In [ ]:
display(llm_with_tools)

이제 클라이언트를 정의한다.

In [21]:
def safe_json_dumps(obj) -> str:
    return json.dumps(obj, ensure_ascii=False, indent=2)


def langchain_mcp_agent(user_message: str, verbose: bool = True) -> dict:
    """
    자연어 요청을 받아 LLM이 필요한 LangChain tool을 직접 선택하게 하는 함수.

    흐름:
    1. 사용자 메시지를 LLM에 보낸다.
    2. LangChain이 tool schema를 모델에 전달한다.
    3. 모델이 필요한 tool name과 arguments를 반환한다.
    4. Python 코드가 해당 LangChain tool을 실행한다.
    5. tool 실행 결과를 ToolMessage로 다시 모델에 넘긴다.
    6. 모델이 최종 답변을 생성한다.
    """

    messages = [
        SystemMessage(
            content=(
                "너는 금융 실습용 assistant다. "
                "제공된 LangChain tools의 name, description, args schema만 근거로 사용자 요청을 처리하라. "
                "특정 tool 이름을 사전에 안다고 가정하지 말고, 제공된 tools 목록에서 적절한 tool을 선택하라. "
                "필수 입력값이 없으면 추측하지 말고 사용자에게 질문하라. "
                "tool 실행 결과를 받으면 한국어로 간결하게 최종 답변하라."
            )
        ),
        HumanMessage(content=user_message),
    ]

    if verbose:
        print("\n" + "=" * 80)
        print("USER:", user_message)
        print("=" * 80)

    # 1차 호출: LLM이 tool을 선택한다.
    ai_msg = llm_with_tools.invoke(messages)
    messages.append(ai_msg)

    tool_calls = ai_msg.tool_calls or []

    executed_tools = []

    if not tool_calls:
        final_answer = ai_msg.content

        if verbose:
            print("\nNO TOOL CALL")
            print("\nFINAL ANSWER:")
            print(final_answer)

        return {
            "user_message": user_message,
            "executed_tools": [],
            "final_answer": final_answer,
        }

    # 2단계: LLM이 선택한 tool을 실제 실행한다.
    for tool_call in tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        if verbose:
            print("\nSELECTED TOOL:", tool_name)
            print("ARGUMENTS:", safe_json_dumps(tool_args))

        if tool_name not in TOOL_MAP:
            tool_result = {
                "error": "UnknownTool",
                "message": f"Unknown tool: {tool_name}",
            }
        else:
            try:
                # LangChain Tool 실행
                tool_result = TOOL_MAP[tool_name].invoke(tool_args)
            except Exception as e:
                tool_result = {
                    "error": type(e).__name__,
                    "message": str(e),
                }

        executed_tools.append({
            "tool_name": tool_name,
            "arguments": tool_args,
            "result": tool_result,
        })

        if verbose:
            print("\nTOOL RESULT:")
            print(safe_json_dumps(tool_result))

        # tool 결과를 LangChain ToolMessage로 대화에 추가
        messages.append(
            ToolMessage(
                content=json.dumps(tool_result, ensure_ascii=False),
                tool_call_id=tool_call_id,
            )
        )

    final_msg = llm.invoke(messages)
    final_answer = final_msg.content

    if verbose:
        print("\nFINAL ANSWER:")
        print(final_answer)

    return {
        "user_message": user_message,
        "executed_tools": executed_tools,
        "final_answer": final_answer,
    }

이제 테스트 해보자. 먼저 계좌와 잔액

In [ ]:
result_1 = langchain_mcp_agent("1001 계좌 잔액 알려줘")
display(result_1)

액수와 조건을 설정해서 조회해 보자

In [ ]:
result_2 = langchain_mcp_agent("1001 계좌에서 3만원 이상 거래만 보여줘")
display(result_2)

모호한 자연어로 테스트해 보자

In [ ]:
result_3 = langchain_mcp_agent("1001 계좌에서 큰돈 나간 거 있어?")
display(result_3)

계좌 전체를 요약시켜 보자

In [ ]:
result_4 = langchain_mcp_agent("1003 계좌 상태 전체적으로 요약해줘. 큰 거래 기준은 5만원으로 봐.")
display(result_4)